In [15]:
import numpy as np
import pandas as pd

import string
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim

In [2]:
df = pd.read_csv("data/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

## Text Preprocessing

In [5]:
# Lower casing

def lower_case(text):
    text = str(text).lower()
    return text

df["review"] = df["review"].apply(lower_case)

In [6]:
# define a function for Text preprocessing

URL_PATTERN = r'https?://\S+|www\.\S+'
HTML_TAGS_PATTERN = r'<.*?>'
PUNCTUATIONS = string.punctuation

STOPWORDS = set(stopwords.words("english"))
stemmer = PorterStemmer()

def text_preprocessing(text):
    # Lower Casing
    text = str(text).lower()

    # URL Remove
    text = re.sub(URL_PATTERN, '', text) # url remove

    # HTML tags remove
    text = re.sub(HTML_TAGS_PATTERN, '', text) # html tags remove
    
    # Punctuations remove
    text = text.translate(str.maketrans("", "", PUNCTUATIONS)) # punctuation remove

    # Stopword & Stemming
    text = " ".join([stemmer.stem(word) for word in str(text).split() if word not in STOPWORDS])
    
    return text

In [7]:
df["review"] = df["review"].apply(text_preprocessing)

## Label Encoding for Sentiment column

In [10]:
lb = LabelEncoder()

df["sentiment"] = lb.fit_transform(df["sentiment"])

## Vectorization - using TFIDF

In [11]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["review"])

In [12]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4048614 stored elements and shape (49582, 5000)>

In [13]:
# because of X type is sparse matrix : we convert it into numpy array
X = X.toarray()

## Prepare our Dataset for Model
**Tensors | Dataset | DataLoaders**

In [20]:
full_dataset = TensorDataset(
    torch.tensor(X, dtype=torch.float32),
    torch.tensor(df["sentiment"].values, dtype=torch.float32))

train_dataset, test_dataset = random_split(full_dataset, [0.8, 0.2])

# build Dataloader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

## Build RNN

In [24]:
class SentimentRNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.5):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN Layer
        self.lstm = nn.LSTM(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True,
            dropout=dropout
        )
        """batch_first=True: It keeps your batch size as the very first dimension (dim=0),
        which makes it much easier to track matrix shapes and debug errors during training."""

        self.dropout = nn.Dropout(dropout)
        
        # fully conected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: [batch_size, seq_len]

        out, _ = self.lstm(x)
        # out => (batch_size, sequence_length, hidden_size)

        out = self.fc(self.dropout(out[:, -1, :])) # out => (batch_size, hidden_size)

        return out

In [25]:
input_size = X_train.shape[1]

model = SentimentRNN(input_size)

criterian = nn.BCELoss()

# add weight_decay(L2 Reg.) for manage Overfittting
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)

## RNN Model Train

In [26]:
epochs=10
train_loss = []
val_loss = []

for epoch in range(epochs):
    model.train()
    running_train_loss = 0.0
    
    for xb, yb in train_loader:
        optimizer.zero_grad()

        xb = xb.unsqueeze(1) # add one more dim in xb beacuse model take 3d
        outputs = model(xb) # model give -  2d: (batch size, 1) => one output for every batch_size value

        # sigmoid take 1d - that's why we do squeeze
        outputs = torch.sigmoid(outputs.squeeze()) # this produce probability

        loss = criterian(outputs, yb)
        loss.backward()
        optimizer.step() # update weights

        running_train_loss += loss.item()

    epoch_train_loss = running_train_loss / len(train_loader)
    train_loss.append(epoch_train_loss)

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.unsqueeze(1)
            outputs = model(xb)
            outputs = torch.sigmoid(outputs.squeeze())
            loss = criterian(outputs, yb)

            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss / len(test_loader)
    val_loss.append(epoch_val_loss)    

    print(f"epoch: {epoch+1}/{epochs} - train_loss={epoch_train_loss} & val_loss={epoch_val_loss}")

epoch: 1/10 - train_loss=0.6816897358625166 & val_loss=0.6210478863408488
epoch: 2/10 - train_loss=0.4114929612846144 & val_loss=0.29745928174065006
epoch: 3/10 - train_loss=0.2825573834440401 & val_loss=0.2707552919464727
epoch: 4/10 - train_loss=0.2594696736263652 & val_loss=0.26405164142770154
epoch: 5/10 - train_loss=0.24551940585576718 & val_loss=0.2619968714252595
epoch: 6/10 - train_loss=0.2361653336834523 & val_loss=0.26238315374620497
epoch: 7/10 - train_loss=0.22930155811050246 & val_loss=0.26389136122119045
epoch: 8/10 - train_loss=0.22196685735496782 & val_loss=0.26595751186532357
epoch: 9/10 - train_loss=0.21744366738825074 & val_loss=0.26823936854639363
epoch: 10/10 - train_loss=0.21406206468180303 & val_loss=0.27065192360070445


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

loss_df = pd.DataFrame({
    "Training Loss": train_loss,
    "Validation Loss": val_loss,
})

plt.plot(loss_df["Training Loss"], label="Training Loss")
plt.plot(loss_df["Validation Loss"], label="Validation Loss")

plt.title("Training vs Validation Loss")
plt.legend()

plt.xlabel("Epochs")
plt.ylabel("Losses")

## Validation

In [98]:
model.eval()
total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb =  xb.unsqueeze(1)
        outputs = model(xb)
        outputs = torch.sigmoid(outputs.squeeze())

        pred = torch.round(outputs)
        
        total += len(yb)
        correct += (pred==yb).sum().item()

print(f"Total data = {total}")
print(f"Correct prediction = {correct}")
print(f"Test Accuracy: {100 * correct / total:.2f}%") 

Total data = 9917
Correct prediction = 8749
Test Accuracy: 88.22%
